# M3의 개선은 어떤 고객에게 갔나 — 고객 가중 M4의 자리가 남아 있는지 (학습 없음)

**⚠ CPU 런타임으로 실행하세요.** H&M 학습이 GPU를 쓰고 있고, 이 진단은 모델당 1초라 GPU가 필요 없습니다.
(런타임 → 런타임 유형 변경 → CPU)

**왜 이걸 먼저 보는가.** M3와 고객 가중 M4를 결합하려면 두 개가 **다른 레버**여야 합니다.
M3는 고객 안에서 어떤 상품 쪽으로 임베딩이 끌려갈지를 바꾸고, 고객 가중 M4는 어느 고객의 오차를
더 크게 볼지를 바꿉니다. 그런데 **M3의 개선이 이미 고q_N 고객에 쏠려 있으면 고객 가중은 중복**입니다.

9/26 축 진단에서 `q_N`은 "거래할 것인가"(Spearman +0.519), `q_V`는 "무엇을 살 것인가"(+0.296)를
예측하고, 순위 평가가 앞의 것을 조건부로 제거함을 확인했습니다. 그 읽기가 맞다면 **M3의 고객별 이득은
q_V를 따라가고 q_N과는 무상관**이어야 하며, 그러면 반복거래축의 자리가 고객 가중에 남습니다.
가정하지 않고 측정합니다.

**판독 기준을 결과 보기 전에 고정했습니다:**
- 세 시드 모두 `|rho| < 0.10` → **고객 가중의 자리가 남아 있다**
- 세 시드 모두 `rho > +0.30` → **M3와 중복**
- 그 사이 / 시드 불일치 → **방향을 말하지 않는다**

**한계:** 재개 파일이 마지막 epoch만 남기므로 **epoch 300 상태**를 읽습니다. arm A의 사전등록 후보는
100 epoch이었습니다. 순위상관은 학습해보지 않은 가중치의 효과가 아니고, 단일 개발분할 결과입니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
M3_DIR = ROOT/'results_v3_dunnhumby_clv_m3_centered_value_graph_v1'
M1_DIR = ROOT/'results_v3_dunnhumby_clv_m2_capacity_search_v1'
if not M3_DIR.is_dir():
    if os.path.ismount('/content/drive'):
        raise RuntimeError(f'Drive는 연결됐지만 M3 결과가 없습니다: {M3_DIR}')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패') from exc
for path in (M3_DIR, M1_DIR):
    if not path.is_dir():
        raise RuntimeError(f'필요한 결과 폴더가 없습니다: {path}')

SOURCE_COMMIT = '53bec51369645c83161812a5920c630370cc8090'
REPO = Path('/content/clv-m3-gain-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), \
        '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import lightgcn_clv_m3_customer_gain_diagnostic as diag
cfg = diag.configure_customer_gain(m3_result_dir=str(M3_DIR), m1_result_dir=str(M1_DIR))
print(json.dumps(diag.preflight_summary(cfg), ensure_ascii=False, indent=2))

## 1. 진단 실행 (학습 없음, 몇 분)
M1·arm A·arm B의 epoch 300 체크포인트를 시드별로 읽어 평가고객 1,230명을 채점합니다.
체크포인트의 커밋·config·epoch을 함께 찍으니, 기대와 다르면 멈추고 확인하세요.
학습은 하지 않으므로 GPU도 필요 없습니다.

In [ ]:
correlation = diag.run_customer_gain(cfg)

## 2. 핵심 표 — 고객별 이득이 어느 축을 따라가나
`quantity=gain`이 M3 이득과 축의 상관, `quantity=m1`은 M1 자체 성과와 축의 상관입니다.
**m1 쪽을 함께 봐야** "고q_N 고객이 애초에 덜 맞춰지고 있었나(=결손이 있나)"를 알 수 있습니다.

In [ ]:
import pandas as pd
pivot = (correlation.pivot_table(index=['model_id', 'quantity', 'metric'],
                                 columns='axis', values='spearman', aggfunc='mean')
         .round(4).reset_index())
print(pivot.to_string(index=False))
print()
print(json.dumps(correlation.attrs['reading'], ensure_ascii=False, indent=2))

## 3. q_N 십분위별 — 결손과 이득이 어디에 있나
`m1_*`이 낮고 `gain_*`이 작은 십분위가 있으면 고객 가중이 손댈 구간입니다.
`improved_share_*`는 그 십분위에서 개선된 고객 비율입니다.

In [ ]:
deciles = diag.decile_frame(correlation)
axis = 'q_n'
view = deciles[deciles[f'{axis}_decile'].notna() & deciles.model_id.eq(diag.ARM_VALUE)]
print(view.groupby(f'{axis}_decile')[
    [f'{axis}_mean', 'm1_recall@10', 'gain_recall@10', 'improved_share_recall@10',
     'm1_revenue@50', 'gain_revenue@50', 'improved_share_revenue@50']
].mean().round(6).to_string())

## 4. 원본 ZIP
고객별 표(1,230명 × 3시드 × 2 arm)까지 들어 있습니다. 상관계수만 보고 결론내지 말고
십분위 표와 M1 결손을 함께 보세요.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
paths = correlation.attrs['result_paths']
zip_path = Path('/content/m3_customer_gain_diagnostic.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))